# PARC2026 — Group-aware Fixed Eval V2

Trajectory Leakage Gate V1で、公開LIBERO-plusの全14,347 episodesが1,681個のnon-visual trajectory groupへまとまり、episode単位holdoutがtrain/eval leakageを起こすことを確認しました。

このNotebookは **2 distinct trajectory groups / task** を固定evalとして選び、各groupから1 episodeだけをeval代表にし、**同groupの全siblingsを全training variantから除外**します。

公開fallbackはData Factory開発用proxyです。Run A固定前には運営 `libero_combined_20hz` でも同じ処理を再実行します。


## Self-contained preflight

`00` / `30` / `40` / `45` を同じruntimeで先に実行していなくても、このNotebook単体で必要なworkspace・repo・parquet・Static Quality・legacy manifestを準備します。


In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys
print('python:', sys.version)
print('platform:', platform.platform())
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
ROOT=Path('/content/parc2026')
for p in [ROOT, ROOT/'vendor', ROOT/'cache', ROOT/'datasets', ROOT/'outputs']:
    p.mkdir(parents=True, exist_ok=True)
REPO=ROOT/'py_AI'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True)
print('repo:', REPO)
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())


## Dataset / Static Qualityを準備

GPUは不要です。運営trajectoryが無ければcompact public `lerobot/libero_plus` v3から **meta + parquetのみ** を取得します。動画はdownloadしません。


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.30','pyarrow>=16','pandas>=2'], check=True)
os.environ.setdefault('HF_HUB_DISABLE_XET','1')
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
PUBLIC_DATASET='lerobot/libero_plus'
PUBLIC_ROOT=ROOT/'datasets'/'public_libero_plus_v3_quality'
ORGANIZER_ROOT=ROOT/'datasets'/'libero_combined_20hz'
configured=os.environ.get('PARC_DATASET_ROOT')
def ready(p): return (p/'meta'/'info.json').exists() and any(p.glob('data/**/*.parquet'))
if configured and ready(Path(configured)):
    DATASET_ROOT=Path(configured); DATASET_ID=os.environ.get('PARC_DATASET_ID','local/libero_combined_20hz'); DATASET_REVISION=None
elif ready(ORGANIZER_ROOT):
    DATASET_ROOT=ORGANIZER_ROOT; DATASET_ID='local/libero_combined_20hz'; DATASET_REVISION=None
else:
    api=HfApi(); ds=api.dataset_info(PUBLIC_DATASET); DATASET_REVISION=ds.sha
    files=api.list_repo_files(PUBLIC_DATASET, repo_type='dataset', revision=DATASET_REVISION)
    required=['meta/info.json','meta/tasks.parquet'] + sorted(f for f in files if f.startswith('data/') and f.endswith('.parquet'))
    for i,filename in enumerate(required,1):
        if i==1 or i==len(required): print(f'download {i}/{len(required)}: {filename}')
        hf_hub_download(repo_id=PUBLIC_DATASET, repo_type='dataset', revision=DATASET_REVISION, filename=filename, local_dir=str(PUBLIC_ROOT))
    DATASET_ROOT=PUBLIC_ROOT; DATASET_ID=PUBLIC_DATASET
print('dataset:', DATASET_ID, '@', DATASET_REVISION)
print('root:', DATASET_ROOT)

STATIC_OUT=ROOT/'outputs'/'static_quality_v1'
METRICS=STATIC_OUT/'episode_quality_metrics.csv'
if not METRICS.exists():
    subprocess.run([sys.executable,str(REPO/'tools/data/static_quality_analyzer.py'),'--root',str(DATASET_ROOT),'--out',str(STATIC_OUT),'--smooth-window','5','--robust-z-threshold','5.0'], check=True)
m=pd.read_csv(METRICS)
print('static:', len(m), 'episodes /', m.task_index.nunique(), 'tasks / review', int(m.quality_review_candidate.sum()))


## Legacy splitを再現し、trajectory group tableを作る

`check_trajectory_group_leakage.py` はparquetから `trajectory_group_members.csv` を作ります。fresh runtimeでは、その入力として旧episode単位manifestを先に再生成します。旧splitがFAILするのは既知なので、ここではfail-fastしません。


In [ ]:
LEGACY_MANIFESTS=ROOT/'outputs'/'dataset_ablation_manifests_v1'
if not (LEGACY_MANIFESTS/'run_matrix.json').exists():
    cmd=[sys.executable,str(REPO/'tools/data/build_dataset_ablation_manifests.py'),'--metrics-csv',str(METRICS),'--out',str(LEGACY_MANIFESTS),'--dataset-id',DATASET_ID,'--seed','20260830','--eval-per-task','2']
    if DATASET_REVISION: cmd += ['--dataset-revision',DATASET_REVISION]
    subprocess.run(cmd, check=True)

LEAK_V1=ROOT/'outputs'/'trajectory_group_leakage_v1'
GROUP_MEMBERS=LEAK_V1/'trajectory_group_members.csv'
if not GROUP_MEMBERS.exists():
    subprocess.run([
        sys.executable,str(REPO/'tools/data/check_trajectory_group_leakage.py'),
        '--root',str(DATASET_ROOT),
        '--manifests-dir',str(LEGACY_MANIFESTS),
        '--out',str(LEAK_V1),
        '--metrics-csv',str(METRICS),
        '--round-decimals','6',
    ], check=True)
legacy=json.loads((LEAK_V1/'trajectory_group_leakage_summary.json').read_text())
print('legacy leakage gate:', legacy['trajectory_leakage_gate'])
print('exact groups:', legacy['exact_group_count'])
print('hashed episodes:', legacy['episode_count_hashed'])


## Group-aware manifest V2を生成

各taskから **異なるexact trajectory groupを2つ** seed固定で選びます。各groupの最小OK episodeをeval代表にし、そのgroupに属する全episodeをtraining poolから除外します。

これによりevalは80 episodesのままですが、train側には同じstate+action trajectoryのvisual/domain siblingを残しません。


In [ ]:
GA_MANIFESTS=ROOT/'outputs'/'dataset_ablation_manifests_v2_group_aware'
cmd=[
    sys.executable,str(REPO/'tools/data/build_group_aware_ablation_manifests.py'),
    '--metrics-csv',str(METRICS),
    '--trajectory-groups-csv',str(GROUP_MEMBERS),
    '--out',str(GA_MANIFESTS),
    '--dataset-id',DATASET_ID,
    '--seed','20260830',
    '--eval-per-task','2',
]
if DATASET_REVISION: cmd += ['--dataset-revision',DATASET_REVISION]
subprocess.run(cmd, check=True)
matrix=json.loads((GA_MANIFESTS/'run_matrix.json').read_text())
eval_manifest=json.loads((GA_MANIFESTS/'FIXED_EVAL_HOLDOUT.json').read_text())
print('group-aware:', matrix['group_aware'])
print('fixed eval:', matrix['fixed_eval']['episode_count'], 'episodes /', matrix['fixed_eval_group_count'], 'groups')
print('protected episodes:', matrix['protected_episode_count'], f"({matrix['protected_fraction']:.2%})")
display(pd.DataFrame([
    {'variant':k,'episodes':v['episode_count'],'frames':v['frame_count'],'tasks':v['task_count']}
    for k,v in matrix['variants'].items()
]))


## Leakage Gateを再実行

このV2 manifestに対してexact trajectory overlapが0であることを実データで再検証します。ここは `--fail-on-exact-leakage` を有効にするため、1件でも残ればセル自体をFAILさせます。


In [ ]:
LEAK_V2=ROOT/'outputs'/'trajectory_group_leakage_v2_group_aware'
subprocess.run([
    sys.executable,str(REPO/'tools/data/check_trajectory_group_leakage.py'),
    '--root',str(DATASET_ROOT),
    '--manifests-dir',str(GA_MANIFESTS),
    '--out',str(LEAK_V2),
    '--metrics-csv',str(METRICS),
    '--round-decimals','6',
    '--fail-on-exact-leakage',
], check=True)
report=json.loads((LEAK_V2/'trajectory_group_leakage_summary.json').read_text())
display(pd.read_csv(LEAK_V2/'manifest_leakage_report.csv'))
assert report['trajectory_leakage_gate']=='PASS'
assert eval_manifest['group_aware'] is True
assert eval_manifest['summary']['episode_count']==2*eval_manifest['summary']['task_count']
assert len(eval_manifest['exact_group_hashes'])==eval_manifest['summary']['episode_count']
print('Group-aware Manifest Gate: PASS')
print('Trajectory Leakage Gate: PASS')


## 次

この2つのPASSを確認してからπ0.5 cheap ablationへ進みます。

次の学習Notebookでは **`dataset_ablation_manifests_v2_group_aware` のみを許可**し、旧episode単位manifestを誤って使わないようguardを入れます。
